# Reproducing the Bank Marketing experiment

This is the primary notebook for reproducing the results currently shown on the ES 335 project website.

**Run this notebook from the repository root** (the directory containing `analysis/`, `data/`, `outputs/`, and `docs/`).

The experiment excludes `duration` because it is not available before a marketing call. It uses a stratified train/test split, training-only cross-validation, fold-specific preprocessing, a majority-class baseline, four fixed candidate classifiers, and a separately tuned Random Forest evaluated on the held-out test set. The held-out test set is not used for model selection.

In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
assert (ROOT / "analysis" / "reproduce_analysis.py").exists(), (
    "Run this notebook from the repository root."
)
assert (ROOT / "data" / "bank-full.csv").exists(), (
    "Place the UCI bank-full.csv file in the data/ directory."
)

print("Python:", sys.version.split()[0])
print("Repository:", ROOT)

Python: 3.13.5
Repository: project root


## 1. Run the reproducible experiment

In [2]:
subprocess.run(
    [sys.executable, "analysis/reproduce_analysis.py"],
    cwd=ROOT,
    check=True
)

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 718, in _warm_feature_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 509, in _warm_drawing_and_image_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/generated/interface/models.py", line 32861, in bring_to_front
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/remote.py", line 764, in __call__
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/client.py", line 175, in call
artifact_tool.rpc.client.Remot

Majority baseline      accuracy=0.8830 F1=0.0000 AP=0.1170 ROC-AUC=0.5000
Logistic Regression    accuracy=0.7553 F1=0.3736 AP=0.4094 ROC-AUC=0.7722
Decision Tree          accuracy=0.7364 F1=0.3381 AP=0.3406 ROC-AUC=0.7175
Random Forest          accuracy=0.8251 F1=0.4476 AP=0.4381 ROC-AUC=0.7976
XGBoost                accuracy=0.8940 F1=0.2537 AP=0.4268 ROC-AUC=0.7807

Saved outputs to: project root/outputs


CompletedProcess(args=['/opt/pyvenv/bin/python', 'analysis/reproduce_analysis.py'], returncode=0)

## 2. Inspect the held-out test comparison

In [3]:
comparison = pd.read_csv(ROOT / "outputs" / "model_comparison.csv")
display(comparison.round(4))

,model,cv_ap_mean,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision,tn,fp,fn,tp,cv_ap_std,cv_roc_auc_mean
0,Majority baseline,0.1170,0.8830,0.5000,0.0000,0.0000,0.0000,0.5000,0.1170,7985,0,1058,0,NaN,NaN
1,Logistic Regression,0.3985,0.7553,0.6983,0.2667,0.6238,0.3736,0.7722,0.4094,6170,1815,398,660,0.0041,0.7633
2,Decision Tree,0.3382,0.7364,0.6666,0.2394,0.5756,0.3381,0.7175,0.3406,6050,1935,449,609,0.0072,0.7206
3,Random Forest,0.4292,0.8251,0.7300,0.3549,0.6059,0.4476,0.7976,0.4381,6820,1165,417,641,0.0032,0.7852
4,XGBoost,0.4206,0.8940,0.5730,0.7181,0.1541,0.2537,0.7807,0.4268,7921,64,895,163,0.0093,0.7762


## 3. Interpret the comparison

The majority-class baseline predicts the negative class for every observation. It can therefore achieve high accuracy on an imbalanced dataset while detecting no subscribers. Compare subscriber precision, recall, F1, balanced accuracy, ROC-AUC, and average precision rather than relying on accuracy alone.

In the current run, the tuned Random Forest has the highest subscriber F1 and ROC-AUC among the evaluated models. XGBoost has the highest accuracy but low subscriber recall at the default 0.50 threshold. These findings describe predictive performance on this dataset and split; they do not establish causal effects.

**Threshold-explorer data source:** The website's threshold explorer reads `outputs/validation_oof_predictions.json`, which contains out-of-fold predictions for the training partition. With three-fold cross-validation, each training observation is scored by a model that did not train on that observation. The explorer recalculates precision, recall, F1 and related classification metrics at user-selected thresholds; it does not fit models in the browser.

**Test-set separation:** `outputs/test_predictions.json` contains predictions for the untouched held-out test partition. These predictions are used for final model evaluation and bootstrap uncertainty analysis, not for interactive threshold exploration or threshold selection. If a threshold is selected for deployment, it should be chosen using training/validation evidence and then evaluated once on the held-out test set. The threshold explorer currently includes the four fixed candidate models; the separately tuned Random Forest is not included because its hyperparameters were selected using training cross-validation, and its displayed OOF scores would require a nested or otherwise selection-aware procedure to avoid optimistic estimates.

## 4. Files generated

- `outputs/model_comparison.csv` — final held-out test metrics for the baseline, four fixed candidates and tuned Random Forest.
- `outputs/test_predictions.json` — held-out test probabilities and labels for final evaluation; **not** used by the threshold explorer.
- `outputs/validation_oof_predictions.json` — out-of-fold predictions from the training partition; this is the data source used by the website threshold explorer.
- `website/data/validation_oof_predictions.json` and `docs/data/validation_oof_predictions.json` — synchronized copies consumed by the source and published websites.
- `outputs/experiment_metadata.json` — split, model and experiment metadata.
- `outputs/test_metric_bootstrap_ci.csv` — 95% percentile bootstrap intervals for held-out test metrics.
- `outputs/random_forest_error_by_group.csv` — descriptive Random Forest error rates across selected groups.
- `outputs/random_forest_grid_search.csv` — Random Forest grid-search results from training-only cross-validation.

The notebook calls `analysis/reproduce_analysis.py` to regenerate the analysis outputs and synchronized website data files. The archived model-specific notebooks are historical references; they are not the source of the current reported results.